# Nested ridge encoding and representational similarity analysis

The notebook orchestrates subject-level feature extraction, nested cross-validation, and RSA.
Core mathematical functions are imported from `src/algonauts_rsa` without changing their
original behavior. Select `smoke`, `fast`, or `full` with `ALGONAUTS_RUN_MODE`; the committed
preliminary results used `fast` (3×2 folds, 200 permutations, 600 RSA images).


In [ ]:
# Optional Colab setup; install the project before running this notebook.
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except ImportError:
    pass


In [ ]:
import json, math, os, random, re, time
from pathlib import Path
from typing import List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import open_clip
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import torchvision.models as tvm
from tqdm import tqdm

from algonauts_rsa.config import get_config, get_paths

RUN_CONFIG = get_config()
PATHS = get_paths()
SEED = RUN_CONFIG.seed
OUTER_FOLDS = RUN_CONFIG.outer_folds
INNER_FOLDS = RUN_CONFIG.inner_folds
ALPHAS = RUN_CONFIG.alphas
N_PERM = RUN_CONFIG.n_permutations
N_RDM_IMAGES = RUN_CONFIG.rsa_subset_size
DEBUG_N_IMAGES = RUN_CONFIG.debug_n_images
SUBJECTS = list(RUN_CONFIG.debug_subjects or [f"subj{idx:02d}" for idx in range(1, 9)])
ROI_KEYS = list(RUN_CONFIG.debug_rois or ["EBA", "FFA", "PPA"])
DEBUG = RUN_CONFIG.name == "smoke"

RESUME = False
FORCE = True
SAVE_PLOTS = False
SAVE_RDM_MATS = False
DO_PERM = N_PERM > 0
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
FEATURE_MODEL = os.getenv("ALGONAUTS_FEATURE_MODEL", "clip")
SAFE_BATCH = 32
NUM_WORKERS = 2
PIN_MEMORY = DEVICE == "cuda"
PERSISTENT_WORKERS = NUM_WORKERS > 0
PREFETCH_FACTOR = 2 if NUM_WORKERS > 0 else None
USE_AMP = True

MULTIROI_BASE = PATHS.multi_roi_dir
RSA_BASE = PATHS.model_results_dir(FEATURE_MODEL)
DATA_ROOT = PATHS.data_root
(RSA_BASE / "features").mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(
    f"mode={RUN_CONFIG.name} | device={DEVICE} | feature={FEATURE_MODEL} | "
    f"folds={OUTER_FOLDS}/{INNER_FOLDS} | permutations={N_PERM} | "
    f"rsa_subset={N_RDM_IMAGES}"
)


In [ ]:
# ==========================================
# Utilities
# ==========================================
from algonauts_rsa.features import extract_clip_features
from algonauts_rsa.rsa import choose_random_indices

class ImageListDataset(Dataset):
    def __init__(self, paths, transform):
        self.paths = paths
        self.transform = transform
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        im = Image.open(self.paths[i]).convert("RGB")
        return self.transform(im)

def load_image_paths(subject: str, n_imgs: int, meta_dir: Path) -> Tuple[List[Path], int]:
    """Load image paths in the CSV order (no shuffle) to keep alignment with Y."""
    meta_path = meta_dir / f"{subject}_train_images_meta.csv"
    if not meta_path.exists():
        raise FileNotFoundError(f"Missing meta CSV: {meta_path}")
    df = pd.read_csv(meta_path)
    n = min(len(df), n_imgs)
    df = df.iloc[:n]
    img_dir = DATA_ROOT / subject / "training_split" / "training_images"
    paths = [img_dir / fn for fn in df["filename"].tolist()]
    return paths, n

# ------- feature extract --------
def extract_resnet_features(paths, batch_size=SAFE_BATCH, tag=""):
    model = tvm.resnet50(weights=tvm.ResNet50_Weights.IMAGENET1K_V1).to(DEVICE).eval()
    tfm = tvm.ResNet50_Weights.IMAGENET1K_V1.transforms()
    feat_extractor = nn.Sequential(*list(model.children())[:-1]).to(DEVICE).eval()

    ds = ImageListDataset(paths, tfm)
    dl_kwargs = dict(
        batch_size=batch_size, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
        persistent_workers=PERSISTENT_WORKERS
    )
    if NUM_WORKERS > 0:
        dl_kwargs["prefetch_factor"] = PREFETCH_FACTOR
    dl = DataLoader(ds, **dl_kwargs)

    feats = []
    amp_ctx = torch.amp.autocast("cuda", enabled=(DEVICE=="cuda" and USE_AMP))
    with torch.no_grad(), amp_ctx:
        for xb in tqdm(dl, desc=f"ResNet feats [{tag}]"):
            xb = xb.to(DEVICE, non_blocking=False)
            f = feat_extractor(xb)     # [B,2048,1,1]
            f = torch.flatten(f, 1)    # [B,2048]
            f = (f / f.norm(dim=-1, keepdim=True)).float().cpu().numpy()
            feats.append(f)
            if DEVICE == "cuda":
                torch.cuda.empty_cache()
    return np.vstack(feats), "ResNet50"

def get_or_make_features(subject: str, n_imgs: int, meta_dir: Path) -> Tuple[np.ndarray, str]:
    """Return (features, tag). Cache under NEW RSA_BASE/features.  FEATURE_MODEL."""
    cache_dir = RSA_BASE / "features"
    cache_dir.mkdir(parents=True, exist_ok=True)

    def try_load(tag):
        p = cache_dir / f"{subject}_{tag}.npy"
        if p.exists():
            F = np.load(p)
            if F.shape[0] >= n_imgs:
                return F[:n_imgs], tag
        return None, None

    # 1) Try cache
    for tag in ["CLIP_ViT-B-32", "ResNet50"]:
        F, used = try_load(tag)
        if F is not None:
            return F, used

    # 2) Compute
    paths, n = load_image_paths(subject, n_imgs, meta_dir)
    if FEATURE_MODEL == "clip":
        F, tag = extract_clip_features(
            paths, batch_size=SAFE_BATCH, tag=subject, device=DEVICE,
            num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
            persistent_workers=PERSISTENT_WORKERS, prefetch_factor=PREFETCH_FACTOR,
            use_amp=USE_AMP,
        )
    elif FEATURE_MODEL == "resnet":
        F, tag = extract_resnet_features(paths, batch_size=SAFE_BATCH, tag=subject)
    else:  # auto
        try:
            F, tag = extract_clip_features(
            paths, batch_size=SAFE_BATCH, tag=subject, device=DEVICE,
            num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
            persistent_workers=PERSISTENT_WORKERS, prefetch_factor=PREFETCH_FACTOR,
            use_amp=USE_AMP,
        )
        except Exception as e:
            print(f"[WARN] CLIP failed ({e}); falling back to ResNet50.")
            F, tag = extract_resnet_features(paths, batch_size=SAFE_BATCH, tag=subject)

    np.save(cache_dir / f"{subject}_{tag}.npy", F)
    return F[:n], tag


In [ ]:
# Encoding implementation is importable and covered by the synthetic smoke test.
from algonauts_rsa.encoding import fast_r2 as _fast_r2, nested_ridge_encoding


In [ ]:
# RSA implementation preserves the original standardization and permutation behavior.
from algonauts_rsa.rsa import compute_rdm, rsa_spearman_perm

def is_done(out_dir: Path):
    required = ["encoding_summary.json", "rsa_summary.json", "r2_per_voxel.npy", "r2_per_voxel_folds.npy"]
    return out_dir.exists() and all((out_dir / name).exists() for name in required)


In [ ]:
import time, json, numpy as np, pandas as pd
from pathlib import Path

# ---------- Logging helpers ----------
VERBOSE = True  # set False to quiet down
def log(msg):
    if VERBOSE:
        print(msg, flush=True)

def stamp():
    return time.perf_counter()

# ======================================
# MAIN LOOP (with progress prints)
# ======================================
plt.rcParams["figure.dpi"] = 130
summary_rows = []

log("== Pipeline start ==")
log(f"Subjects: {len(SUBJECTS)} | ROIs: {len(ROI_KEYS)} | RESUME={RESUME} FORCE={FORCE} | DEBUG={DEBUG} DEBUG_N_IMAGES={DEBUG_N_IMAGES}")
log(f"Paths: MULTIROI_BASE={MULTIROI_BASE} | RSA_BASE={RSA_BASE}")

total_t0 = stamp()
n_pairs_total = 0

for si, subj in enumerate(SUBJECTS, start=1):
    subj_t0 = stamp()
    log(f"\n--- [SUBJECT {si}/{len(SUBJECTS)}] {subj} ---")

    meta_dir = MULTIROI_BASE / "meta"
    subj_meta = meta_dir / f"{subj}_train_images_meta.csv"
    if not subj_meta.exists():
        log(f"[SKIP] {subj}: meta CSV not found -> {subj_meta}")
        continue

    # probe ROI to get subject n_images
    probe_roi = next((r for r in ROI_KEYS if (MULTIROI_BASE / r / f"{subj}.npy").exists()), None)
    if probe_roi is None:
        log(f"[SKIP] {subj}: no ROI arrays found in any ROI among {list(ROI_KEYS)}")
        continue

    Y_probe_path = MULTIROI_BASE / probe_roi / f"{subj}.npy"
    log(f"[{subj}] Probe ROI: {probe_roi} -> {Y_probe_path}")
    Y_probe = np.load(Y_probe_path, mmap_mode="r")
    n_imgs_subj = Y_probe.shape[0]
    if DEBUG and DEBUG_N_IMAGES is not None:
        n_imgs_subj = min(n_imgs_subj, DEBUG_N_IMAGES)
    log(f"[{subj}] n_images (after DEBUG cap if any): {n_imgs_subj}")

    # features once per subject
    t0 = stamp()
    log(f"[{subj}] Getting features via get_or_make_features(...).")
    F_full, feat_tag = get_or_make_features(subj, n_imgs_subj, meta_dir)
    F_full = F_full[:n_imgs_subj]
    log(f"[{subj}] Features ready: F_full shape={F_full.shape} | feature_model={feat_tag} | took {stamp()-t0:.2f}s")

    # ---------- Randomized RSA subset (subject-level), with alignment ----------
    if N_RDM_IMAGES:
        n_req = min(N_RDM_IMAGES, n_imgs_subj)
        log(f"[{subj}] Choosing RSA subset: requested={N_RDM_IMAGES}, effective={n_req}")
        rdm_idx_subj = choose_random_indices(n_imgs_subj, n_req, seed=SEED)
    else:
        rdm_idx_subj = np.arange(n_imgs_subj)
        log(f"[{subj}] Using all images for RSA subset: {len(rdm_idx_subj)}")

    # Precompute feature RDM for the chosen subset (same order as rdm_idx_subj)
    t0 = stamp()
    F_rdm = F_full[rdm_idx_subj]
    log(f"[{subj}] Computing feature RDM on F_rdm shape={F_rdm.shape}")
    rdm_feat_subj = compute_rdm(F_rdm, metric="cosine", zscore_columns=True)
    log(f"[{subj}] Feature RDM done, shape={rdm_feat_subj.shape} | took {stamp()-t0:.2f}s")

    # Keep a record of the indices used for RSA at the subject level
    subj_rdm_idx_dir = RSA_BASE / "rdm_indices" / subj
    subj_rdm_idx_dir.mkdir(parents=True, exist_ok=True)
    np.save(subj_rdm_idx_dir / "rdm_indices.npy", rdm_idx_subj)
    log(f"[{subj}] Saved RSA subset indices -> {subj_rdm_idx_dir/'rdm_indices.npy'}")

    for ri, roi in enumerate(ROI_KEYS, start=1):
        roi_file = MULTIROI_BASE / roi / f"{subj}.npy"
        if not roi_file.exists():
            log(f"[SKIP] {subj} {roi}: ROI array missing -> {roi_file}")
            continue

        out_dir = RSA_BASE / roi / subj
        (out_dir / "plots").mkdir(parents=True, exist_ok=True)

        if RESUME and not FORCE and is_done(out_dir):
            log(f"[RESUME] skip {subj} {roi} (already done) -> {out_dir}")
            continue

        log(f"[RUN ] {subj} {roi} | out_dir={out_dir}")
        pair_t0 = stamp()

        # --- Load Y and align ---
        Y = np.load(roi_file)
        if DEBUG and DEBUG_N_IMAGES is not None:
            Y = Y[:n_imgs_subj]

        n = min(F_full.shape[0], Y.shape[0])
        F = F_full[:n]
        Y = Y[:n]
        log(f"[{subj} {roi}] Shapes aligned: F={F.shape}, Y={Y.shape}")

        # --- Encoding ---
        t0 = stamp()
        log(f"[{subj} {roi}] Encoding start: OUTER_FOLDS={OUTER_FOLDS}, INNER_FOLDS={INNER_FOLDS}, |ALPHAS|={len(ALPHAS)}")
        mean_r2, chosen_alphas, fold_r2 = nested_ridge_encoding(
            F, Y, OUTER_FOLDS, INNER_FOLDS, ALPHAS, SEED
        )
        enc_dur = stamp() - t0
        k_top = max(1, int(0.10 * Y.shape[1]))
        top10_mean = float(np.mean(np.partition(mean_r2, -k_top)[-k_top:]))
        enc_mean = float(np.nanmean(mean_r2))
        enc_median = float(np.nanmedian(mean_r2))
        log(f"[{subj} {roi}] Encoding done in {enc_dur:.2f}s | r2_mean={enc_mean:.4f}, r2_median={enc_median:.4f}, r2_top10_mean={top10_mean:.4f}")

        np.save(out_dir / "r2_per_voxel.npy", mean_r2)
        np.save(out_dir / "r2_per_voxel_folds.npy", fold_r2)
        with open(out_dir / "encoding_summary.json", "w") as f:
            json.dump({
                "subject": subj,
                "roi": roi,
                "n_images": int(n),
                "n_voxels": int(Y.shape[1]),
                "feature_model": feat_tag,
                "outer_folds": OUTER_FOLDS,
                "inner_folds": INNER_FOLDS,
                "alphas": list(map(float, ALPHAS)),
                "chosen_alphas": list(map(float, chosen_alphas)),
                "r2_mean": enc_mean,
                "r2_median": enc_median,
                "r2_top10_mean": top10_mean
            }, f, indent=2)
        log(f"[{subj} {roi}] Saved encoding outputs.")

        if SAVE_PLOTS:
            log(f"[{subj} {roi}] Saving encoding histogram plot...")
            plt.figure()
            plt.hist(mean_r2, bins=40)
            plt.title(f"Ridge encoding R² — {subj} {roi} ({feat_tag})\n"
                      f"mean={enc_mean:.3f}, median={enc_median:.3f}")
            plt.xlabel("R² per voxel"); plt.ylabel("Count")
            plt.tight_layout()
            plt.savefig(out_dir / "plots" / "encoding_r2_hist.png")
            plt.close()

        # --- RSA (use the SAME randomized indices, filtered to within n) ---
        idx_eff = rdm_idx_subj[rdm_idx_subj < n]
        n_rdm_eff = len(idx_eff)
        log(f"[{subj} {roi}] RSA subset effective size: {n_rdm_eff} (<= n={n})")

        t0 = stamp()
        Y_rdm = Y[idx_eff]
        rdm_fmri = compute_rdm(Y_rdm, metric="correlation", zscore_columns=True)
        rdm_feat_eff = rdm_feat_subj[:n_rdm_eff, :n_rdm_eff]
        rsa_build_dur = stamp() - t0
        log(f"[{subj} {roi}] Built fMRI/feature RDMs in {rsa_build_dur:.2f}s | shapes fmri={rdm_fmri.shape}, feat={rdm_feat_eff.shape}")

        t0 = stamp()
        if DO_PERM and N_PERM > 0:
            log(f"[{subj} {roi}] RSA Spearman with permutations: N_PERM={N_PERM}")
            rho, pval = rsa_spearman_perm(rdm_fmri, rdm_feat_eff, n_perm=N_PERM, seed=SEED)
        else:
            iu = np.triu_indices_from(rdm_fmri, k=1)
            rho, _ = spearmanr(rdm_fmri[iu], rdm_feat_eff[iu])
            pval = float("nan")
        rsa_stat_dur = stamp() - t0
        log(f"[{subj} {roi}] RSA done in {rsa_stat_dur:.2f}s | rho={rho:.4f}, p={pval}")

        if SAVE_RDM_MATS:
            np.save(out_dir / "rdm_fmri.npy", rdm_fmri)
            np.save(out_dir / "rdm_feat.npy", rdm_feat_eff)
            np.save(out_dir / "rdm_indices_used.npy", idx_eff)
            log(f"[{subj} {roi}] Saved RDM matrices + indices.")

        with open(out_dir / "rsa_summary.json", "w") as f:
            json.dump({
                "subject": subj, "roi": roi,
                "feature_model": feat_tag,
                "rho_spearman": float(rho),
                "p_value_perm": float(pval),
                "n_perm": int(N_PERM if DO_PERM else 0),
                "n_rdm_images": int(n_rdm_eff)
            }, f, indent=2)
        log(f"[{subj} {roi}] Saved RSA summary.")

        if SAVE_PLOTS:
            log(f"[{subj} {roi}] Saving RSA plots...")
            fig = plt.figure(figsize=(5,4))
            plt.imshow(rdm_fmri, interpolation="none"); plt.colorbar()
            plt.title(f"RDM fMRI — {subj} {roi}")
            plt.tight_layout(); plt.savefig(out_dir / "plots" / "rdm_fmri.png"); plt.close(fig)

            fig = plt.figure(figsize=(5,4))
            plt.imshow(rdm_feat_eff, interpolation="none"); plt.colorbar()
            plt.title(f"RDM Features — {subj} ({feat_tag})")
            plt.tight_layout(); plt.savefig(out_dir / "plots" / "rdm_feat.png"); plt.close(fig)

            iu = np.triu_indices_from(rdm_fmri, k=1)
            plt.figure(figsize=(4.2,4.2))
            plt.scatter(rdm_feat_eff[iu], rdm_fmri[iu], s=6, alpha=0.5)
            plt.xlabel("Feature distances"); plt.ylabel("fMRI distances")
            plt.title(f"RSA: ρ={rho:.3f}, p={pval:.4f}")
            plt.tight_layout(); plt.savefig(out_dir / "plots" / "rsa_scatter.png"); plt.close()

        summary_rows.append({
            "subject": subj,
            "roi": roi,
            "feature_model": feat_tag,
            "n_images": n,
            "n_voxels": Y.shape[1],
            "r2_mean": enc_mean,
            "r2_median": enc_median,
            "r2_top10_mean": top10_mean,
            "rsa_rho": float(rho),
            "rsa_p": float(pval)
        })
        n_pairs_total += 1
        log(f"[{subj} {roi}] Pair complete in {stamp()-pair_t0:.2f}s.")

    log(f"--- [{subj}] Subject complete in {stamp()-subj_t0:.2f}s ---")

# save global summary
if summary_rows:
    df_sum = pd.DataFrame(summary_rows)
    RSA_BASE.mkdir(parents=True, exist_ok=True)
    df_sum.to_csv(RSA_BASE / "summary_rsa.csv", index=False)
    log(f"\n== DONE: wrote {len(df_sum)} rows to {RSA_BASE/'summary_rsa.csv'} ==")
    log(f"Total (subj,roi) pairs processed: {n_pairs_total} | total time {stamp()-total_t0:.2f}s")
    try:
        from IPython.display import display
        display(df_sum)
    except Exception:
        print(df_sum.head())
else:
    log("\n== No results produced. Check paths/DEBUG/RESUME/FORCE settings. ==")
